In [ ]:
import os
import numpy as np
import pandas as pd

# Relative to Our Notebooks/
PROVIDED_DIR = "../Provided Datasets"
NEW_DIR = "../New Datasets"
OUTPUT_DIR = NEW_DIR

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Provided dir:", os.path.abspath(PROVIDED_DIR))
print("New dir     :", os.path.abspath(NEW_DIR))
print("Output dir  :", os.path.abspath(OUTPUT_DIR))


In [ ]:
# Expected training files (we will look in BOTH folders for each file)
EXPECTED_FILES = {
    "gaia": "gaia_features_training.csv",
    "jrc_gsw": "jrc_gsw_features_training.csv",
    "landsat_allbands": "landsat_features_training_allbands.csv",
    "terraclimate_allvars": "terraclimate_features_training_allvars.csv",
    "esa_cci": "esa_cci_features_training.csv",
    "water_quality": "water_quality_training_dataset.csv",
    # These two may be duplicates/reduced versions; include them if present
    "terraclimate": "terraclimate_features_training.csv",
    "landsat": "landsat_features_training.csv",
}

def resolve_path(fname: str) -> str | None:
    """Return the first existing path for fname across Provided and New dirs."""
    for base in (PROVIDED_DIR, NEW_DIR):
        p = os.path.join(base, fname)
        if os.path.exists(p):
            return p
    return None

resolved = {}
missing = []
for key, fname in EXPECTED_FILES.items():
    p = resolve_path(fname)
    if p is None:
        missing.append((key, fname))
    else:
        resolved[key] = p

print("Resolved input paths:")
for k, p in resolved.items():
    print(f" - {k:18s} -> {os.path.abspath(p)}")

if missing:
    msg = "Missing these expected files in BOTH folders:\n" + "\n".join([f"{k}: {f}" for k, f in missing])
    raise FileNotFoundError(msg)

print("\nAll expected files found")


Resolved input paths:
 - gaia               -> /Users/ethanchan/Desktop/Water-Quality-Prediction/New Datasets/gaia_features_training.csv
 - jrc_gsw            -> /Users/ethanchan/Desktop/Water-Quality-Prediction/New Datasets/jrc_gsw_features_training.csv
 - landsat_allbands   -> /Users/ethanchan/Desktop/Water-Quality-Prediction/New Datasets/landsat_features_training_allbands.csv
 - terraclimate_allvars -> /Users/ethanchan/Desktop/Water-Quality-Prediction/New Datasets/terraclimate_features_training_allvars.csv
 - esa_cci            -> /Users/ethanchan/Desktop/Water-Quality-Prediction/New Datasets/esa_cci_features_training.csv
 - water_quality      -> /Users/ethanchan/Desktop/Water-Quality-Prediction/Provided Datasets/water_quality_training_dataset.csv
 - terraclimate       -> /Users/ethanchan/Desktop/Water-Quality-Prediction/Provided Datasets/terraclimate_features_training.csv
 - landsat            -> /Users/ethanchan/Desktop/Water-Quality-Prediction/Provided Datasets/landsat_features_t

In [ ]:
def standardize_join_keys(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize join columns to: latitude, longitude, sample_date."""
    out = df.copy()
    rename_map = {}
    for c in out.columns:
        lc = c.strip().lower()
        if lc in {"latitude", "lat"} or lc.startswith("lat"):
            rename_map[c] = "latitude"
        elif lc in {"longitude", "lon", "lng"} or lc.startswith("lon"):
            rename_map[c] = "longitude"
        # Be conservative: only map obvious sample-date columns
        elif lc in {"sample date", "sample_date"}:
            rename_map[c] = "sample_date"
    out = out.rename(columns=rename_map)

    # If a dataset used a generic "Date" column, map it ONLY if sample_date doesn't exist yet
    if "sample_date" not in out.columns:
        for c in list(out.columns):
            if c.strip().lower() == "date":
                out = out.rename(columns={c: "sample_date"})
                break

    # Drop duplicate columns created by renaming collisions
    out = out.loc[:, ~out.columns.duplicated(keep="first")]

    required = {"latitude", "longitude", "sample_date"}
    missing = required - set(out.columns)
    if missing:
        raise ValueError(f"Missing required join columns after standardization: {missing}")

    # helper parsed date (not used for join)
    out["sample_date_parsed"] = pd.to_datetime(out["sample_date"], errors="coerce", infer_datetime_format=True)
    return out


In [ ]:
# Load + standardize
datasets = {}
for name, path in resolved.items():
    df = pd.read_csv(path)
    df_std = standardize_join_keys(df)
    datasets[name] = df_std
    print(f"{name:18s} shape={df_std.shape}  cols={len(df_std.columns)}")


gaia               shape=(9319, 9)  cols=9
jrc_gsw            shape=(9319, 13)  cols=13
landsat_allbands   shape=(9319, 16)  cols=16
terraclimate_allvars shape=(9319, 18)  cols=18
esa_cci            shape=(9319, 9)  cols=9
water_quality      shape=(9319, 7)  cols=7
terraclimate       shape=(9319, 5)  cols=5
landsat            shape=(9319, 10)  cols=10


/var/folders/v5/7p0nb6rn16l5rcpvjdqmtkj80000gn/T/ipykernel_49892/1560012526.py:32: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  out["sample_date_parsed"] = pd.to_datetime(out["sample_date"], errors="coerce", infer_datetime_format=True)
/var/folders/v5/7p0nb6rn16l5rcpvjdqmtkj80000gn/T/ipykernel_49892/1560012526.py:32: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  out["sample_date_parsed"] = pd.to_datetime(out["sample_date"], errors="coerce", infer_datetime_format=True)
/var/folders/v5/7p0nb6rn16l5rcpvjdqmtkj80000gn/T/ipykernel_49892/1560012526.py:32: UserW

In [ ]:
# Outer merge on join keys
merge_keys = ["latitude", "longitude", "sample_date"]

merged = None
for name, df in datasets.items():
    if merged is None:
        merged = df
    else:
        merged = pd.merge(
            merged,
            df,
            on=merge_keys,
            how="outer",
            suffixes=("", f"__{name}")  # helps prevent _x/_y
        )

print("Merged shape:", merged.shape)
merged.head()


Merged shape: (9319, 66)


,latitude,longitude,sample_date,gaia_changed_ever_frac,gaia_impervious_frac_by_sample_year,gaia_recent_change_5y_frac,gaia_years_since_change_mean,gaia_transition_year_mean_changed_pixels,sample_date_parsed,Total Alkalinity,...,sample_date_parsed__water_quality,pet__terraclimate,sample_date_parsed__terraclimate,nir__landsat,green__landsat,swir16__landsat,swir22__landsat,NDMI__landsat,MNDWI__landsat,sample_date_parsed__landsat
0,-34.405833,19.600556,01-10-2014,0.0,0.0,0.0,-1.0,-1.0,2014-01-10,54.181,...,2014-01-10,173.0,2014-01-10,19002.5,9353.0,13580.0,10717.0,0.166424,-0.184320,2014-01-10
1,-34.405833,19.600556,02-08-2011,0.0,0.0,0.0,-1.0,-1.0,2011-02-08,36.247,...,2011-02-08,174.1,2011-02-08,NaN,NaN,NaN,NaN,NaN,NaN,2011-02-08
2,-34.405833,19.600556,02-12-2015,0.0,0.0,0.0,-1.0,-1.0,2015-02-12,57.800,...,2015-02-12,163.0,2015-02-12,NaN,NaN,NaN,NaN,NaN,NaN,2015-02-12
3,-34.405833,19.600556,03-07-2013,0.0,0.0,0.0,-1.0,-1.0,2013-03-07,29.719,...,2013-03-07,162.8,2013-03-07,14964.0,8714.0,11536.5,9401.0,0.129337,-0.139379,2013-03-07
4,-34.405833,19.600556,03-09-2014,0.0,0.0,0.0,-1.0,-1.0,2014-03-09,58.231,...,2014-03-09,173.0,2014-03-09,17562.5,9283.5,13263.5,10589.0,0.139460,-0.176520,2014-03-09


In [ ]:
# Save merged + two filled versions
raw_out = os.path.join(OUTPUT_DIR, "combined_training_dataset.csv")
merged.to_csv(raw_out, index=False)

# Version 1: Drop rows with ANY null values
dropped_na = merged.dropna()

drop_out = os.path.join(OUTPUT_DIR, "combined_training_dataset_dropna.csv")
dropped_na.to_csv(drop_out, index=False)

print("Wrote:", os.path.abspath(drop_out))
print("Original rows:", len(merged))
print("Rows after dropna:", len(dropped_na))


# Version 2: numeric nulls -> column mean
filled_mean = merged.copy()
numeric_cols = filled_mean.select_dtypes(include=[np.number]).columns
for c in numeric_cols:
    m = filled_mean[c].mean()
    if pd.notna(m):
        filled_mean[c] = filled_mean[c].fillna(m)
mean_out = os.path.join(OUTPUT_DIR, "combined_training_dataset_nulls_as_mean.csv")
filled_mean.to_csv(mean_out, index=False)

print("Wrote:")
print(" -", os.path.abspath(raw_out))
print(" -", os.path.abspath(drop_out))
print(" -", os.path.abspath(mean_out))


Wrote: /Users/ethanchan/Desktop/Water-Quality-Prediction/New Datasets/combined_training_dataset_dropna.csv
Original rows: 9319
Rows after dropna: 1220
Wrote:
 - /Users/ethanchan/Desktop/Water-Quality-Prediction/New Datasets/combined_training_dataset.csv
 - /Users/ethanchan/Desktop/Water-Quality-Prediction/New Datasets/combined_training_dataset_dropna.csv
 - /Users/ethanchan/Desktop/Water-Quality-Prediction/New Datasets/combined_training_dataset_nulls_as_mean.csv


In [ ]:
# Quick missing-value counts
def na_count(df):
    return int(df.isna().sum().sum())

print("Total missing values:")
print("  merged (raw):", na_count(merged))
print("  dropped_na  :", na_count(dropped_na))
print("  filled_mean :", na_count(filled_mean), "(non-numeric NaNs may remain)")

print("\nRow counts:")
print("  merged (raw):", len(merged))
print("  dropped_na  :", len(dropped_na))
print("  filled_mean :", len(filled_mean))


Total missing values:
  merged (raw): 77741
  dropped_na  : 0
  filled_mean : 43816 (non-numeric NaNs may remain)

Row counts:
  merged (raw): 9319
  dropped_na  : 1220
  filled_mean : 9319


In [ ]:
# Count nulls per column
null_counts = merged.isna().sum()

# Sort descending
null_counts_sorted = null_counts.sort_values(ascending=False)

# Show top 10 columns with most nulls
print("Top 10 columns with most nulls:")
print(null_counts_sorted.head(10))

# Get the single column with the most nulls
max_null_column = null_counts_sorted.idxmax()
max_null_count = null_counts_sorted.max()

print("\nColumn with MOST nulls:")
print("Column:", max_null_column)
print("Number of nulls:", max_null_count)
print("Percent null:", round(100 * max_null_count / len(merged), 2), "%")


Top 10 columns with most nulls:
lwir11                                  6418
coastal                                 6161
qa_aerosol                              6156
sample_date_parsed__landsat             5477
sample_date_parsed                      5477
sample_date_parsed__esa_cci             5477
sample_date_parsed__water_quality       5477
sample_date_parsed__jrc_gsw             5477
sample_date_parsed__terraclimate        5477
sample_date_parsed__landsat_allbands    5477
dtype: int64

Column with MOST nulls:
Column: lwir11
Number of nulls: 6418
Percent null: 68.87 %


In [ ]:
# --------------------------------------------------
# Remove all extra sample_date_parsed columns
# (keep only the base 'sample_date_parsed')
# --------------------------------------------------

cols_to_drop = [
    c for c in merged.columns
    if c.startswith("sample_date_parsed") and c != "sample_date_parsed"
]

print("Dropping parsed-date columns:")
print(cols_to_drop)

merged = merged.drop(columns=cols_to_drop)

print("New shape after dropping parsed-date columns:", merged.shape)
